# Daily Practice — 2026-10-11 — Statistics for ML Quality: Bootstrap Confidence Intervals & Drift Detection

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) — 569 real tumor samples, each with 30 numeric features computed from digitized images of fine needle aspirate (FNA) biopsies, labeled malignant/benign. Loaded via `sklearn.datasets.load_breast_cancer` (a real diagnostic dataset bundled with scikit-learn, not a synthetic toy).

**Problem.** A single number like "test accuracy = 95%" is a point estimate, not a verdict — the same way one passing test run doesn't prove a feature is stable under load. A QA mindset applied to a shipped model should always ask two questions: (1) how much would this metric move if we re-sampled the test set? and (2) does the population you're scoring in "production" still look like the population you validated on? In this notebook you'll train a simple classifier on the real Breast Cancer Wisconsin data, then answer both questions with core statistics tools: bootstrap resampling for confidence intervals, and two-sample hypothesis testing for distribution drift.

**What you should produce:**
1. `bootstrap_ci(y_true, y_score, metric_fn, n_boot=2000, ci=0.95)` — a general-purpose bootstrap confidence interval function, applied to get 95% CIs for both the model's test accuracy and its ROC-AUC.
2. `ks_drift_report(reference_df, current_df, alpha=0.05)` — runs a two-sample Kolmogorov-Smirnov test feature-by-feature between two DataFrames, returning a results table with each feature's KS statistic, p-value, and significance flags under both the raw `alpha` and a Bonferroni-corrected alpha (since you're testing 30 features at once, not one).
3. Apply `ks_drift_report` to compare the real test set against a deliberately biased *slice* of that same test set (rows where `worst radius` is above its 75th percentile) — a stand-in for a production population skewed toward larger, more severe tumors. Report which features show drift that survives Bonferroni correction, and which only looked "significant" at the raw alpha=0.05 threshold.
4. A short write-up (3–5 sentences): given the CI width on your metrics and the drift results, would you ship this model as-is, and what would you specifically monitor in production?

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42
pd.set_option("display.width", 120)

## Load the data and train a baseline model

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target  # 0 = malignant, 1 = benign
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

pred_test = model.predict(X_test)
proba_test = model.predict_proba(X_test)[:, 1]

test_accuracy = accuracy_score(y_test, pred_test)
test_auc = roc_auc_score(y_test, proba_test)
print(f"Point-estimate accuracy: {test_accuracy:.4f}")
print(f"Point-estimate ROC-AUC:  {test_auc:.4f}")

## Your task

Fill in the TODOs below. Function signatures are given; the logic is up to you.

In [ ]:
def bootstrap_ci(y_true, y_score, metric_fn, n_boot=2000, ci=0.95, random_state=RANDOM_STATE):
    """Compute a bootstrap confidence interval for metric_fn(y_true, y_score).

    Resample (y_true, y_score) pairs WITH replacement `n_boot` times, computing
    metric_fn on each resample, then return (point_estimate, lower, upper) where
    lower/upper are the percentiles of the bootstrap distribution corresponding
    to the requested `ci` (e.g. ci=0.95 -> 2.5th and 97.5th percentiles).

    y_true, y_score: 1D arrays/Series of the same length, already aligned
        (y_score can be hard predictions or probabilities -- metric_fn decides).
    metric_fn: callable(y_true_sample, y_score_sample) -> float
    """
    # TODO:
    # 1. Convert y_true, y_score to numpy arrays.
    # 2. Create a reproducible RNG: rng = np.random.default_rng(random_state).
    # 3. For n_boot iterations: draw len(y_true) indices WITH replacement
    #    (rng.integers or rng.choice), compute metric_fn on the resampled
    #    (y_true, y_score) pair, and collect the results into a list.
    # 4. Compute the point estimate by calling metric_fn on the *original*
    #    (unresampled) y_true, y_score.
    # 5. Return (point_estimate, lower_percentile, upper_percentile), where the
    #    percentiles come from np.percentile on the bootstrap distribution using
    #    the tails implied by `ci` (e.g. ci=0.95 -> 2.5 and 97.5).
    raise NotImplementedError

In [ ]:
# TODO: call bootstrap_ci twice:
#   - once with metric_fn = accuracy_score, using pred_test as y_score
#   - once with metric_fn = roc_auc_score, using proba_test as y_score
# Print both point estimates with their 95% CI, e.g.:
#   "Accuracy: 0.9474 [0.9064, 0.9766]"
#   "ROC-AUC:  0.9912 [0.9780, 0.9981]

## Drift detection

Now check whether a "production-like" slice of the data still looks statistically
like the validation population.

In [ ]:
def ks_drift_report(reference_df, current_df, alpha=0.05):
    """Run a two-sample Kolmogorov-Smirnov test on each column shared by
    reference_df and current_df.

    Returns a DataFrame with columns: feature, ks_stat, p_value,
    significant_raw, significant_bonferroni -- sorted by p_value ascending.

    significant_raw: p_value < alpha
    significant_bonferroni: p_value < (alpha / number_of_features_tested)
    """
    # TODO:
    # 1. For each column shared by reference_df and current_df, run
    #    scipy.stats.ks_2samp(reference_df[col], current_df[col]).
    # 2. Collect feature name, the KS statistic, and the p-value into a list
    #    of dicts (or parallel lists).
    # 3. Build a DataFrame from the collected results. Compute the
    #    Bonferroni-corrected alpha as alpha / (number of features tested),
    #    then add the two boolean significance columns.
    # 4. Sort by p_value ascending and return the DataFrame.
    raise NotImplementedError

In [ ]:
# TODO:
# 1. Build `biased_slice`: the rows of X_test where X_test["worst radius"] is
#    above its 75th percentile -- a stand-in for a production population
#    skewed toward larger, more severe tumors.
# 2. Call ks_drift_report(X_test, biased_slice) and store the result as
#    `drift_report`.
# 3. Print drift_report, and separately print how many features are
#    significant_raw vs significant_bonferroni.

### Write-up

TODO: in 3–5 sentences, say (a) how wide your bootstrap CIs are and whether
that changes how much you trust the point-estimate accuracy/AUC, (b) how many
features drift under Bonferroni correction vs. the raw threshold and why that
gap matters when you're scanning 30 features instead of 1, and (c) what you
would actually monitor in production given these results.

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def bootstrap_ci(y_true, y_score, metric_fn, n_boot=2000, ci=0.95, random_state=RANDOM_STATE):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n = len(y_true)
    rng = np.random.default_rng(random_state)

    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_stats[i] = metric_fn(y_true[idx], y_score[idx])

    point_estimate = metric_fn(y_true, y_score)
    tail = (1 - ci) / 2 * 100
    lower, upper = np.percentile(boot_stats, [tail, 100 - tail])
    return point_estimate, lower, upper


acc_point, acc_lo, acc_hi = bootstrap_ci(y_test.to_numpy(), pred_test, accuracy_score)
auc_point, auc_lo, auc_hi = bootstrap_ci(y_test.to_numpy(), proba_test, roc_auc_score)
print(f"Accuracy: {acc_point:.4f} [{acc_lo:.4f}, {acc_hi:.4f}]")
print(f"ROC-AUC:  {auc_point:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")
```

```python
def ks_drift_report(reference_df, current_df, alpha=0.05):
    shared_cols = [c for c in reference_df.columns if c in current_df.columns]
    rows = []
    for col in shared_cols:
        stat, pvalue = stats.ks_2samp(reference_df[col], current_df[col])
        rows.append({"feature": col, "ks_stat": stat, "p_value": pvalue})

    report = pd.DataFrame(rows)
    bonferroni_alpha = alpha / len(shared_cols)
    report["significant_raw"] = report["p_value"] < alpha
    report["significant_bonferroni"] = report["p_value"] < bonferroni_alpha
    return report.sort_values("p_value").reset_index(drop=True)


biased_slice = X_test[X_test["worst radius"] > X_test["worst radius"].quantile(0.75)]

drift_report = ks_drift_report(X_test, biased_slice)
print(drift_report)
print()
print(f"significant_raw:        {drift_report['significant_raw'].sum()} / {len(drift_report)}")
print(f"significant_bonferroni: {drift_report['significant_bonferroni'].sum()} / {len(drift_report)}")
```

**Explanation.** `bootstrap_ci` resamples the paired `(y_true, y_score)` arrays
with replacement `n_boot` times, recomputes the metric on each resample, and
reads the confidence bounds off the percentiles of that resampled
distribution — this is the nonparametric bootstrap, and it works for *any*
metric without assuming a particular sampling distribution for it.

`ks_drift_report` runs one Kolmogorov-Smirnov two-sample test per feature. The
KS test compares the empirical CDFs of a feature in two populations and is
sensitive to any difference in distribution (location, spread, or shape), not
just a shift in the mean. Because `worst radius` is explicitly the slicing
variable, it (and features strongly correlated with tumor size, like `mean
radius`, `mean perimeter`, `worst perimeter`, `worst area`) show the strongest
drift and survive Bonferroni correction — 17 of 30 features do. A second
group (`mean texture`, `compactness error`, `worst texture`, `mean symmetry`)
clears the raw `alpha=0.05` threshold but not the Bonferroni one: their
p-values are only weakly significant, driven by a faint, indirect correlation
with tumor size rather than a strong shift of their own. That 21-vs-17 gap is
exactly why a monitoring pipeline comparing dozens of features at once needs a
multiple-testing correction — without it, every scan would keep escalating
borderline features instead of just the handful that are genuinely, strongly
shifted.

</details>